# Model Export & On-Device Deployment — Core ML

Notebook ini mengubah model forecasting menjadi format Core ML (`.mlpackage`) agar dapat berjalan langsung di perangkat iOS tanpa server. Model produksi dibangun ulang dengan layer standar Keras dalam bentuk *single-shot* — memprediksi 24 langkah dalam satu forward pass — agar kompatibel dengan converter Core ML dan efisien untuk inferensi on-device.

Arsitektur eksperimental dengan custom layer dan decoder autoregresif didokumentasikan pada notebook riset terpisah; notebook ini fokus pada jalur deployment yang siap dikirim ke produksi.

**Output:** `BitcoinForecaster.mlpackage` dan `scaler_params.json`.

## 1. Setup

In [1]:
!pip install -q coremltools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00


In [2]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
import coremltools as ct

print("TensorFlow :", tf.__version__)
print("coremltools:", ct.__version__)

TensorFlow : 2.20.0
coremltools: 9.0


## 2. Data Loading & Preprocessing

Pipeline preprocessing identik dengan notebook riset: feature engineering rolling statistics, split kronologis 70/15/15 yang dilakukan sebelum normalisasi untuk mencegah kebocoran data, dan normalisasi MinMax yang di-fit hanya pada data train.

In [3]:
csv_url = 'https://drive.google.com/uc?export=download&id=1hpsqSpfjdqIZWqwd259klQSeaNSe5Trr'
df = pd.read_csv(csv_url)
df['Date'] = pd.to_datetime(df['Date'], format='mixed')
df = df.sort_values('Date').reset_index(drop=True)
df.set_index('Date', inplace=True)

df['Close_Rolling_Mean_24'] = df['Close'].rolling(window=24).mean()
df['Close_Rolling_Std_24']  = df['Close'].rolling(window=24).std()
df.dropna(inplace=True)

print("Shape:", df.shape)

Shape: (53127, 8)


In [4]:
WINDOW_SIZE = 48
HORIZON     = 24
TARGET_COL  = 'Close'
BATCH_SIZE  = 64
feature_columns = ['Close', 'Volume USDT', 'RSI', 'ATR',
                   'Close_Rolling_Mean_24', 'Close_Rolling_Std_24']
N_FEATURES  = len(feature_columns)

data = df[feature_columns].values
n = len(data)
train_end, val_end = int(n * 0.70), int(n * 0.85)
train_data = data[:train_end]
val_data   = data[train_end:val_end]
test_data  = data[val_end:]

scalers = {}
train_scaled = np.zeros_like(train_data, dtype=np.float32)
val_scaled   = np.zeros_like(val_data,   dtype=np.float32)
test_scaled  = np.zeros_like(test_data,  dtype=np.float32)

for i, col in enumerate(feature_columns):
    sc = MinMaxScaler()
    train_scaled[:, i] = sc.fit_transform(train_data[:, i].reshape(-1, 1)).flatten()
    val_scaled[:, i]   = sc.transform(val_data[:, i].reshape(-1, 1)).flatten()
    test_scaled[:, i]  = sc.transform(test_data[:, i].reshape(-1, 1)).flatten()
    scalers[col] = sc

print(f"Train {train_data.shape[0]} | Val {val_data.shape[0]} | Test {test_data.shape[0]}")

Train 37188 | Val 7969 | Test 7970


In [5]:
def create_windows(data, window_size, horizon, target_idx=0):
    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i : i + window_size])
        y.append(data[i + window_size : i + window_size + horizon, target_idx])
    return np.array(X), np.array(y)


target_idx = feature_columns.index(TARGET_COL)
X_train, y_train = create_windows(train_scaled, WINDOW_SIZE, HORIZON, target_idx)
X_val,   y_val   = create_windows(val_scaled,   WINDOW_SIZE, HORIZON, target_idx)
X_test,  y_test  = create_windows(test_scaled,  WINDOW_SIZE, HORIZON, target_idx)

train_dataset = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
                 .shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset   = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
                 .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset  = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
                 .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print(f"X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}")

X_train (37117, 48, 6) | X_val (7898, 48, 6) | X_test (7899, 48, 6)


## 3. Production Model (Standard Keras Layers)

Model dibangun dengan Keras Functional API menggunakan layer bawaan (`LSTM`, `MultiHeadAttention`, `LayerNormalization`, `Dense`). Bentuk single-shot dipilih karena lebih ringan dan jauh lebih andal dikonversi ke Core ML dibanding decoder autoregresif.

In [6]:
def build_production_model(window_size=WINDOW_SIZE, n_features=N_FEATURES,
                           horizon=HORIZON, unroll=False):
    inputs = keras.Input(shape=(window_size, n_features), name="price_window")

    x = layers.LSTM(128, return_sequences=True, unroll=unroll)(inputs)
    x = layers.Dropout(0.2)(x)

    attn = layers.MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = layers.LayerNormalization()(x + attn)

    x = layers.LSTM(64, unroll=unroll)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(horizon, name="forecast")(x)

    return keras.Model(inputs, outputs, name="bitcoin_forecaster_production")


prod_model = build_production_model()
prod_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mae")
prod_model.summary()

Model: "bitcoin_forecaster_production"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ price_window        │ (None, 48, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 48, 128)   │     69,120 │ price_window[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 48, 128)   │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 48, 128)   │     66,048 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 48, 128)   │          0 │ dropout[0][0],    │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 48, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 64)        │     49,408 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,160 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forecast (Dense)    │ (None, 24)        │      1,560 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 190,552 (744.34 KB)

 Trainable params: 190,552 (744.34 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Training

In [7]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5),
]

history = prod_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    callbacks=callbacks,
    verbose=1,
)

prod_test_mae = prod_model.evaluate(test_dataset, verbose=0)
print(f"Production model - Test MAE (scaled): {prod_test_mae:.6f}")

prod_model.save("model_production.keras")

Epoch 1/50
580/580 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - loss: 0.0243 - val_loss: 0.0424 - learning_rate: 0.0010
Epoch 2/50
580/580 ━━━━━━━━━━━━━━━━━━━━ 18s 14ms/step - loss: 0.0185 - val_loss: 0.0224 - learning_rate: 0.0010
Epoch 3/50
580/580 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - loss: 0.0137 - val_loss: 0.0198 - learning_rate: 0.0010
Epoch 4/50
580/580 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - loss: 0.0128 - val_loss: 0.0273 - learning_rate: 0.0010
Epoch 5/50
580/580 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - loss: 0.0117 - val_loss: 0.0308 - learning_rate: 0.0010
Epoch 6/50
580/580 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - loss: 0.0112 - val_loss: 0.0240 - learning_rate: 0.0010
Epoch 7/50
580/580 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - loss: 0.0098 - val_loss: 0.0237 - learning_rate: 5.0000e-04
Epoch 8/50
580/580 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - loss: 0.0096 - val_loss: 0.0355 - learning_rate: 5.0000e-04
Epoch 9/50
580/580 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - loss: 0.0098 - val_loss: 0.0231 - learning_rate

## 5. Core ML Conversion

Layer LSTM dengan kernel cuDNN (default ketika training di GPU) tidak didukung converter Core ML. Untuk konversi, arsitektur yang identik dibangun ulang dengan `unroll=True` — secara matematis setara namun menghasilkan operasi standar yang dapat dikonversi — lalu bobot hasil training disalin ke model tersebut.

Hasilnya berformat ML Program (`.mlpackage`) dengan target deployment iOS 16. Input berupa window 48 jam x 6 fitur ter-normalisasi; output berupa 24 nilai prediksi dalam ruang ter-normalisasi yang di-inverse-scale di sisi aplikasi.

In [8]:
prod_model = keras.models.load_model("model_production.keras")

conv_model = build_production_model(unroll=True)
conv_model.set_weights(prod_model.get_weights())

mlmodel = ct.convert(
    conv_model,
    inputs=[ct.TensorType(name="price_window",
                          shape=(1, WINDOW_SIZE, N_FEATURES),
                          dtype=np.float32)],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS16,
)

mlmodel.short_description = "Bitcoin 24-hour hourly price forecaster (single-shot, on-device)"
mlmodel.author = "Muhammad Fariz Abizar"
mlmodel.save("BitcoinForecaster.mlpackage")

spec = mlmodel.get_spec()
print("Inputs :", [i.name for i in spec.description.input])
print("Outputs:", [o.name for o in spec.description.output])
print("Saved  : BitcoinForecaster.mlpackage")

Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 39.60 passes/s]


Inputs : ['price_window']
Outputs: ['Identity']
Saved  : BitcoinForecaster.mlpackage


## 6. Scaler Export

Parameter normalisasi (min/max per fitur) diekspor ke JSON agar aplikasi iOS dapat melakukan scaling input dan inverse-scaling output secara konsisten dengan proses training.

In [9]:
scaler_params = {
    "feature_columns": feature_columns,
    "target_col": TARGET_COL,
    "window_size": WINDOW_SIZE,
    "horizon": HORIZON,
    "scalers": {
        col: {
            "min": float(scalers[col].data_min_[0]),
            "max": float(scalers[col].data_max_[0]),
        }
        for col in feature_columns
    },
}

with open("scaler_params.json", "w") as f:
    json.dump(scaler_params, f, indent=2)

print(json.dumps(scaler_params, indent=2))

{
  "feature_columns": [
    "Close",
    "Volume USDT",
    "RSI",
    "ATR",
    "Close_Rolling_Mean_24",
    "Close_Rolling_Std_24"
  ],
  "target_col": "Close",
  "window_size": 48,
  "horizon": 24,
  "scalers": {
    "Close": {
      "min": 3172.05,
      "max": 68633.69
    },
    "Volume USDT": {
      "min": 0.0,
      "max": 1514464825.2185037
    },
    "RSI": {
      "min": 35.178834150766306,
      "max": 64.33650355472459
    },
    "ATR": {
      "min": 17.436265687318684,
      "max": 1004.5314074329413
    },
    "Close_Rolling_Mean_24": {
      "min": 3212.5458333333336,
      "max": 67566.14333333333
    },
    "Close_Rolling_Std_24": {
      "min": 4.474908604005059,
      "max": 3493.4577866880923
    }
  }
}


## 7. Parity Check (macOS)

Verifikasi bahwa output model Core ML identik dengan model Keras asli. Sel ini memerlukan macOS karena `MLModel.predict()` hanya tersedia di sana; lewati jika dijalankan di Colab/Linux.

In [10]:
sample = X_test[:1].astype(np.float32)

keras_out  = conv_model.predict(sample, verbose=0)[0]
pred       = mlmodel.predict({"price_window": sample})
out_key    = list(pred.keys())[0]
coreml_out = np.array(pred[out_key]).flatten()

diff = np.mean(np.abs(keras_out - coreml_out))
print(f"Output key    : {out_key}")
print(f"Mean abs diff : {diff:.8f}")
print("Parity OK" if diff < 1e-3 else "Mismatch - investigate")

Exception: Model prediction is only supported on macOS version 10.13 or later.

In [11]:
import os, zipfile
from google.colab import files

artifacts = [
    # Phase 1
    "model_baseline_LSTM.keras",
    "model_seq2seq_LSTM.keras",
    "best_model_seq2seq_LSTM.keras",
    "shap_beeswarm.png",
    "shap_importance_bar.png",
    "shap_waterfall.png",
    "shap_horizon_comparison.png",
    "model_comparison_final.png",
    # Phase 2
    "model_production.keras",
    "scaler_params.json",
]

zip_name = "btc_artifacts.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in artifacts:
        if os.path.exists(f):
            zf.write(f)
            print("added:", f)
    for folder in ["BitcoinForecaster.mlpackage", "mlruns"]:
        if os.path.isdir(folder):
            for root, _, names in os.walk(folder):
                for n in names:
                    zf.write(os.path.join(root, n))
            print("added:", folder + "/")

print(f"\nZipped -> {zip_name}")
files.download(zip_name)

added: model_production.keras
added: scaler_params.json
added: BitcoinForecaster.mlpackage/

Zipped -> btc_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>